<img src="https://raw.githubusercontent.com/MohammedAly22/qwencleo-asr/main/assets/QwenCleo-ASR-Banner.png" width="100%"/>

# 🎙️ QwenCleo-ASR — vLLM Serving & True Streaming

Egyptian Arabic & code-switching speech recognition, built on Qwen3-ASR.

[Model](https://huggingface.co/mohammedaly22/QwenCleo-ASR) · [GitHub](https://github.com/MohammedAly22/qwencleo-asr) · [PyPI](https://pypi.org/project/qwencleo-asr/)

> **Runtime → Change runtime type → GPU** before running. Then run the cells top to bottom.

QwenCleo inherits Qwen3-ASR's **token-by-token streaming** via vLLM.

> ⚠️ Needs a recent CUDA. On Colab pick a **GPU** runtime; vLLM's Qwen3-ASR support is on the **nightly** build, which is large — the install cell can take several minutes.

## 1. Install vLLM (nightly) + the client

`--index-strategy` is a `uv` flag (not pip), so we install `uv` first and use it for the nightly wheel.

In [ ]:
# vLLM nightly has day-0 Qwen3-ASR support
!pip install -q uv
!uv pip install --system -q -U vllm --pre \
    --extra-index-url https://wheels.vllm.ai/nightly \
    --index-strategy unsafe-best-match
!uv pip install --system -q 'vllm[audio]' openai httpx
!pip install -q qwencleo-asr --no-deps

## 2. Download the model first

Pull weights now so the server boots fast in the next cell.

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('mohammedaly22/QwenCleo-ASR')
print('model downloaded')

## 3. Start a vLLM server in the background

In [ ]:
import subprocess, time, requests
proc = subprocess.Popen(['vllm', 'serve', 'mohammedaly22/QwenCleo-ASR'])
print('starting vLLM server...')
for _ in range(120):
    try:
        requests.get('http://localhost:8000/v1/models', timeout=2)
        print('server up'); break
    except Exception:
        if proc.poll() is not None:
            raise RuntimeError('vLLM exited — check logs above')
        time.sleep(10)

## 4. Sample audio

In [ ]:
# Grab a sample Egyptian/code-switching clip to transcribe
import urllib.request, os
URL = 'https://huggingface.co/mohammedaly22/QwenCleo-ASR/resolve/main/assets/sample.wav'
FALLBACK = 'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav'
path = 'sample.wav'
try:
    urllib.request.urlretrieve(URL, path)
except Exception:
    urllib.request.urlretrieve(FALLBACK, path)
print('saved', path, os.path.getsize(path), 'bytes')

## 5. TRUE streaming — deltas arrive token by token

In [ ]:
from qwencleo_asr import stream_vllm

for delta in stream_vllm('sample.wav', language='Arabic'):
    print(delta, end='', flush=True)
print()

## 6. One-shot via the OpenAI-compatible API

In [ ]:
from qwencleo_asr import transcribe_vllm
print(transcribe_vllm('sample.wav'))

## 7. OpenAI SDK directly (the documented form)

In [ ]:
from openai import OpenAI
client = OpenAI(base_url='http://localhost:8000/v1', api_key='EMPTY')
with open('sample.wav','rb') as f:
    print(client.audio.transcriptions.create(
        model='mohammedaly22/QwenCleo-ASR', file=f.read()).text)

## 8. Stop the server

In [ ]:
proc.terminate()